In [ ]:
from pathlib import import Path
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from etl.common import init_spark3

import sys
import time
import importlib
import pandas as pd

BASE_DIR = Path("/apps/jupyter/users/nguyetnguyen/features")

sys.path[:0] = [str(path) for path in [
    BASE_DIR / "common" / "src",
    BASE_DIR / "device_refactored",
]]

from common.merge_features import merge_features_from_sample_path
from common.utils import dir_ls
import configs.configs as config

importlib.reload(config)
spark = init_spark3.setup(
    job_cfg={
        'executor.instances': 8,
        'executor.cores': 8,
        'executor.memory': '20g',
    },
    script_name="sample_preparation"
)

# Create base df

## From feature store

In [ ]:
pd_all_path_scan = pd.read_csv('/apps/jupyter/users/nguyentguyen/template/fit_model/ver2/output/model_build_dataset_catalog_selected.csv')[['dataset_path',
'dataset_group']].sort_values(by='dataset_group').reset_index(drop=True)
all_path_features = pd_all_path_scan['dataset_path'].to_list()
sorted(all_path_features)

In [ ]:
label_path = '/label/client=hc/source_code=20250417_hc_train_credit_score_81k'
output_path = '/user/nguyentguyen/tmp/test_device_feature/merge_test/hc_2025'
del_list_features = [
    '/score/model_code=vnpt_cs_bnpl_v2',
    '/score/model_code=vnpt_cs_cash_loan_v1',
    '/score/model_code=vnpt_cs_generic_v6',
    '/score/model_code=vnpt_cs_hc_generic_v5',
    '/score/model_code=vnpt_cs_hc_generic_v6',
    '/score/model_code=vnpt_cs_kv_v2'
]

# df_label = spark.read.parquet(label_path)

# for feature_path in all_path_features:

#     if feature_path in del_list_features:
#         continue

#     merge_each_feature_from_feature_store(
#         spark=spark,
#         df_label=df_label,
#         feature_path=feature_path,
#         output_base_path=output_path,
#         prefix='hc_2025'
#     )

# df_final, final_path = merge_all_features(
#     spark=spark,
#     label_path=label_path,
#     feature_dir=output_path,
#     output_dir=output_path,
#     prefix='hc_2025',
#     del_paths=del_list_features
# )

## From sample path 

In [ ]:
label_path = '/label/client=hc/source_code=20250417_hc_train_credit_score_81k'
sample_path = '/project/vnpt_cs_hc_v6_fix/feature/join_feature'
output_path = '/user/nguyetnguyen/tmp/test_new_features/base_df'
all_path_sample_features = dir_ls(sample_path, 'dico_hc_hd_kv_tp_random_377k_pn_date')
all_path_sample_features

In [ ]:
del_paths = []
# df_final = merge_features_from_sample_path(spark, all_path_sample_features, del_paths, label_path, output_path)
# df_final.write.mode('overwrite').parquet(config.BASE_FEATURES_PATH)

# pd.DataFrame({'feature': sorted(spark.read.parquet(config.BASE_FEATURES_PATH).columns)}).to_csv('./all_features.csv')

# Create imei weekly sample HC 2025

In [ ]:
for snapshot_date in pd.date_range('2022-06-01', '2024-10-31', freq='W-SUN'):
    start_time = time.time()
    snapshot_date_str = snapshot_date.strftime('%Y-%m-%d')
    print(datetime.now(), snapshot_date_str)
    
    df_data = (spark.read.parquet(f'{config.IMEI_WEEKLY_PATH}/date={snapshot_date_str}')
               .join(spark.read.parquet('/user/nguyetnguyen/features/hc_2025/phone_number'),
                     on='phone_number',
                     how='inner'
                    )
              )
    
    df_data.write.mode('overwrite').parquet(f'{config.IMEI_WEEKLY_HC_SAMPLE_PATH}/date={snapshot_date_str}')
    
    print('Done at: ', datetime.now())
    print('-'*50)